# NLST — Clean External Evaluation

Loads per-fold predictions, mean slide aggregation, computes ZS + FT C-index + HR.
Saves results to `results/external_results/nlst.json`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
import warnings; warnings.filterwarnings('ignore')

from sparc.utils.external_eval import (
    load_perfold_slides, evaluate_cohort,
    print_results, save_results
)

In [ ]:
import os
# ── Config ──
SQ_DIR  = Path('results/nlst_inference/mmp20_sq_cos20_knn64_es_20260303_132721_perfold')
IMG_DIR = Path('results/nlst_inference/mmp20_image_only_cos20_es_20260303_130951_perfold')
DICOM_CSV = os.environ.get('SPARC_NLST_DICOM_CSV', 'data/nlst_dicom_file_list.csv')
LUAD_MORPHS = {8140, 8250, 8255, 8260, 8265, 8480, 8490, 8310, 8323}
LUSC_MORPHS = {8070, 8071, 8072, 8052, 8083}

MODELS = {
    'sq': 'mmp20_sq_cos20_knn64_es_20260303_132721',
    'img': 'mmp20_image_only_cos20_es_20260303_130951',
}

In [ ]:
# ── Load slides ──
sq_slides = load_perfold_slides(SQ_DIR)
img_slides = load_perfold_slides(IMG_DIR)
print(f'Loaded {len(sq_slides)} SQ, {len(img_slides)} IMG slides')

# ── UUID -> PID mapping ──
dicom = pd.read_csv(DICOM_CSV)
dicom['uuid'] = dicom['file_name'].str.replace('.dcm', '', regex=False)
dicom['pid'] = dicom['directory'].str.extract(r'/nlst/(\d+)/').astype(int)
uuid_to_pid = dict(zip(dicom['uuid'], dicom['pid']))

# ── Group slides by patient ──
pid_slides = defaultdict(lambda: {'sq': [], 'img': []})
for stem in sq_slides:
    if stem not in img_slides:
        continue
    pid = uuid_to_pid.get(stem)
    if pid is None:
        continue
    pid_slides[pid]['sq'].append(sq_slides[stem])
    pid_slides[pid]['img'].append(img_slides[stem])

print(f'{len(pid_slides)} patients with slides')

In [ ]:
# ── Clinical ──
nlst = pd.read_csv('data/nlst_outcomes/nlst.csv', low_memory=False)
canc = pd.read_csv('data/nlst_outcomes/nlst_780_canc_idc_20210527.csv')[['pid', 'lc_morph']].drop_duplicates('pid')

nl = nlst[nlst['pid'].isin(pid_slides)].copy()
nl['os_time'] = nl.apply(
    lambda r: r['death_days'] - r['candx_days'] if pd.notna(r['death_days'])
    else r['fup_days'] - r['candx_days'], axis=1)
nl['os_event'] = (nl['deathstat'] == 1).astype(int)
nl['dss_time'] = nl['os_time']
nl['dss_event'] = nl['finaldeathlc'].fillna(0).astype(int)
nl = nl[(nl['deathstat'] != 2) & (nl['progressed_ever'] != 9)]
nl = nl[(nl['os_time'] > 0) & nl['os_time'].notna()]
nl = nl.merge(canc, on='pid', how='left')

def get_histo(m):
    if pd.isna(m): return 'Other'
    m = int(m)
    return 'LUAD' if m in LUAD_MORPHS else ('LUSC' if m in LUSC_MORPHS else 'Other')
nl['histo'] = nl['lc_morph'].apply(get_histo)
print(f'{len(nl)} patients after filtering (LUAD={sum(nl.histo=="LUAD")}, LUSC={sum(nl.histo=="LUSC")})')

In [ ]:
# ── Slide aggregation ──
# Toggle: 'mean' or 'max_risk'
AGGREGATION = 'mean'

rows = []
for _, r in nl.iterrows():
    pid = int(r['pid'])
    slides_sq = pid_slides[pid]['sq']
    slides_img = pid_slides[pid]['img']

    sq_avg_risks = [s['risk'].mean() for s in slides_sq]
    img_avg_risks = [s['risk'].mean() for s in slides_img]

    if AGGREGATION == 'mean':
        risk_sq = float(np.mean(sq_avg_risks))
        risk_img = float(np.mean(img_avg_risks))
        emb_sq = np.stack([s['embedding'].mean(axis=0) for s in slides_sq]).mean(axis=0)
        emb_img = np.stack([s['embedding'].mean(axis=0) for s in slides_img]).mean(axis=0)
    elif AGGREGATION == 'max_risk':
        # Sensitivity option: pick the slide with highest per-slide ensembled risk.
        sq_best = np.argmax(sq_avg_risks)
        img_best = np.argmax(img_avg_risks)
        risk_sq = float(sq_avg_risks[sq_best])
        risk_img = float(img_avg_risks[img_best])
        emb_sq = slides_sq[sq_best]['embedding'].mean(axis=0)
        emb_img = slides_img[img_best]['embedding'].mean(axis=0)
    else:
        raise ValueError(f'Unknown aggregation: {AGGREGATION}')

    rows.append({
        'pid': pid, 'T_os': r['os_time'], 'E_os': float(r['os_event']),
        'T_dss': r['dss_time'], 'E_dss': float(r['dss_event']),
        'histo': r['histo'],
        'risk_sq': risk_sq, 'risk_img': risk_img,
        'emb_sq': emb_sq, 'emb_img': emb_img,
    })

df = pd.DataFrame(rows)
print(f'{len(df)} patients ({AGGREGATION} aggregation)')

In [ ]:
# ── Evaluate ──
endpoints = [('DSS', 'T_dss', 'E_dss'), ('OS', 'T_os', 'E_os')]

res_all = evaluate_cohort(df, endpoints, label='All')
print_results(res_all)

save_results('nlst', [res_all], MODELS,
             'results/external_results/nlst.json')